<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Assignment_8_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),                         # convert image to tensor
    transforms.Normalize((0.5, 0.5, 0.5),          # normalize each channel
                         (0.5, 0.5, 0.5))
])

train_full = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


100%|██████████| 170M/170M [04:39<00:00, 609kB/s]


In [4]:
train_labels = np.array(train_full.targets)   # get labels as numpy array

# indices for classes 0–4 (Subset A) and 5–9 (Subset B)
indices_A = np.where((train_labels >= 0) & (train_labels <= 4))[0]
indices_B = np.where((train_labels >= 5) & (train_labels <= 9))[0]


In [5]:
def make_balanced_indices(indices, labels, class_list, per_class=3000):
    balanced = []
    for c in class_list:
        class_idx = indices[labels[indices] == c]   # all indices of class c
        class_idx = class_idx[:per_class]           # take first N samples
        balanced.extend(class_idx)
    return balanced

balanced_A = make_balanced_indices(indices_A, train_labels, [0,1,2,3,4], per_class=3000)
balanced_B = make_balanced_indices(indices_B, train_labels, [5,6,7,8,9], per_class=3000)

subsetA = Subset(train_full, balanced_A)
subsetB = Subset(train_full, balanced_B)


In [6]:
from torch.utils.data import Dataset, DataLoader, Subset, random_split
val_ratio = 0.2
val_size = int(len(subsetA) * val_ratio)
train_size = len(subsetA) - val_size

trainA, valA = random_split(subsetA, [train_size, val_size])


In [7]:
batch_size = 64

train_loaderA = DataLoader(trainA, batch_size=batch_size, shuffle=True)
val_loaderA = DataLoader(valA, batch_size=batch_size, shuffle=False)


In [9]:
import torch.nn as nn
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 32x16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 64 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 64x8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)                  # 5 classes: 0–4
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=5).to(device)


In [11]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [12]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loaderA:
        images = images.to(device)
        labels = labels.to(device)

        # labels are 0–4 already, no remapping needed

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loaderA:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch [1/10] Train Loss: 1.0702, Train Acc: 0.5583 Val Loss: 0.9636, Val Acc: 0.5893
Epoch [2/10] Train Loss: 0.8180, Train Acc: 0.6733 Val Loss: 0.7680, Val Acc: 0.6993
Epoch [3/10] Train Loss: 0.6963, Train Acc: 0.7292 Val Loss: 0.7212, Val Acc: 0.7280
Epoch [4/10] Train Loss: 0.6142, Train Acc: 0.7648 Val Loss: 0.6692, Val Acc: 0.7393
Epoch [5/10] Train Loss: 0.5471, Train Acc: 0.7915 Val Loss: 0.6369, Val Acc: 0.7633
Epoch [6/10] Train Loss: 0.4908, Train Acc: 0.8142 Val Loss: 0.6131, Val Acc: 0.7787
Epoch [7/10] Train Loss: 0.4281, Train Acc: 0.8404 Val Loss: 0.6419, Val Acc: 0.7720
Epoch [8/10] Train Loss: 0.3594, Train Acc: 0.8698 Val Loss: 0.6775, Val Acc: 0.7557
Epoch [9/10] Train Loss: 0.3099, Train Acc: 0.8863 Val Loss: 0.6654, Val Acc: 0.7797
Epoch [10/10] Train Loss: 0.2495, Train Acc: 0.9104 Val Loss: 0.7193, Val Acc: 0.7653
